In [2]:
import os

DATA_PATH = None

for root, dirs, files in os.walk("/kaggle/input"):
    if "standardized_256" in dirs:
        DATA_PATH = os.path.join(root, "standardized_256")
        break

print("데이터 경로:", DATA_PATH)
print("클래스:", sorted(os.listdir(DATA_PATH)))

데이터 경로: /kaggle/input/datasets/sumn2u/garbage-classification-v2/standardized_256
클래스: ['battery', 'biological', 'cardboard', 'clothes', 'glass', 'metal', 'paper', 'plastic', 'shoes', 'trash']


In [3]:
import os
import shutil
import random
from pathlib import Path

random.seed(42)

SOURCE_DIR = Path(DATA_PATH)
OUTPUT_DIR = Path("/kaggle/working/garbage_split")

train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for class_dir in SOURCE_DIR.iterdir():
    if not class_dir.is_dir():
        continue

    images = [
        file for file in class_dir.iterdir()
        if file.suffix.lower() in image_extensions
    ]

    random.shuffle(images)

    total = len(images)
    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)

    split_files = {
        "train": images[:train_end],
        "val": images[train_end:val_end],
        "test": images[val_end:]
    }

    for split, files in split_files.items():
        destination = OUTPUT_DIR / split / class_dir.name
        destination.mkdir(parents=True, exist_ok=True)

        for file in files:
            shutil.copy2(file, destination / file.name)

    print(
        class_dir.name,
        "전체:", total,
        "train:", len(split_files["train"]),
        "val:", len(split_files["val"]),
        "test:", len(split_files["test"])
    )

print("완료:", OUTPUT_DIR)

metal 전체: 930 train: 744 val: 93 test: 93
glass 전체: 1736 train: 1388 val: 173 test: 175
biological 전체: 699 train: 559 val: 69 test: 71
paper 전체: 1336 train: 1068 val: 133 test: 135
battery 전체: 756 train: 604 val: 75 test: 77
trash 전체: 453 train: 362 val: 45 test: 46
cardboard 전체: 1411 train: 1128 val: 141 test: 142
shoes 전체: 1449 train: 1159 val: 144 test: 146
clothes 전체: 1892 train: 1513 val: 189 test: 190
plastic 전체: 1597 train: 1277 val: 159 test: 161
완료: /kaggle/working/garbage_split


In [5]:
!pip install -q ultralytics

from ultralytics import YOLO

model = YOLO("yolo11n-cls.pt")

model.train(
    data="/kaggle/working/garbage_split",
    epochs=50,
    imgsz=256,
    batch=32,
    device=0,
    patience=10,
    project="/kaggle/working/runs",
    name="garbage_yolo11n"
)

Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/garbage_split, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=garbage_yolo11n-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, 

Exception in thread Thread-34 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd
    fd = df.detach()
         ^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/resource_

KeyboardInterrupt: 